# BƯỚC 2: CHỨNG MINH DỮ LIỆU UY TÍN 

File `aqi_hcm_daily.csv` bạn vừa tạo ở Bước 1 là data thật, nhưng **"thật" chưa đủ**. Notebook này kiểm tra 6 việc, mỗi việc trả lời 1 câu hỏi khác nhau.


In [1]:
import requests
import pandas as pd
import numpy as np

DATA = "../01_Data/aqi_hcm_daily.csv"
LAT, LON = 10.78533, 106.67029          # dung lai toa do tram CMT8, de goi CAMS o Truc cot 4

df = pd.read_csv(DATA, parse_dates=["date"])   # doc lai dung file vua luu o Buoc 1
n = len(df)
print(f"Da doc {n} dong tu {DATA}")
print(f"Khoang thoi gian: {df['date'].min().date()} -> {df['date'].max().date()}")

Da doc 657 dong tu ../01_Data/aqi_hcm_daily.csv
Khoang thoi gian: 2024-11-19 -> 2026-10-19


## Bằng chứng 1: Nguồn gốc minh bạch

**Câu hỏi trả lời:** data này lấy từ đâu, ai xác nhận lại được? Một dữ liệu đáng tin trước hết phải **truy được nguồn**.

- **Nguồn 1 — OpenAQ** (tổ chức phi lợi nhuận quốc tế), trạm CMT8, Location ID 3276359, TP.HCM. Kiểm chứng: https://explore.openaq.org/locations/3276359
- **Nguồn 2 — Open-Meteo Archive API**, tái phân tích ERA5/ECMWF (châu Âu). Tài liệu: https://open-meteo.com/en/docs/historical-weather-api

→ Ai cũng mở link kiểm tra được.

## Bằng chứng 2: Chất lượng đo lường

**Câu hỏi:** dữ liệu có thiếu nhiều không, có giá trị bất thường (outlier) không?

**Quy tắc IQR:** `IQR = Q3 − Q1` (độ trải rộng của 50% dữ liệu ở giữa); giá trị thấp hơn `Q1 − 1,5·IQR` hoặc cao hơn `Q3 + 1,5·IQR` coi là outlier.

In [2]:
print("[2] CHAT LUONG DO LUONG")
print("-" * 60)

cov = df["pm25_cov"]
print(f"Do phu TB (% gio/ngay co do)   : {cov.mean():.1f}%")
print(f"So ngay phu >= 75%             : {(cov >= 75).sum()}/{n} ({100*(cov>=75).mean():.1f}%)")

# isna(): kiem tra tung o co rong khong -> sum() dem tong so o rong tren ca 5 cot chinh
thieu = df[["pm25", "temperature", "humidity", "wind", "rain"]].isna().sum().sum()
print(f"Tong so o bi thieu              : {thieu}  (chi tinh cac dong dang co, KHONG tinh ngay vang mat)")

# Ngay bi mat han (khong co dong nao): dem tren lich lien tuc tu ngay dau den ngay cuoi
ngay_thieu = (df["date"].max() - df["date"].min()).days + 1 - n
print(f"So ngay VANG MAT khoi bang      : {ngay_thieu}  (2 khoang tram mat du lieu, xem notebook 00)")
print(f"So ngay cam bien chi do < 50% gio: {(cov < 50).sum()}  (trung binh ngay dua tren rat it gio)")

q1, q3 = df.pm25.quantile([.25, .75])   # quantile(): tim gia tri tai dung mot phan tram cho truoc
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
n_out = ((df.pm25 < lo) | (df.pm25 > hi)).sum()   # dem so dong PM2.5 nam ngoai [lo, hi]
print(f"Nguong binh thuong (IQR)        : [{lo:.1f}, {hi:.1f}]")
print(f"So ngay ngoai lai               : {n_out}/{n} ({100*n_out/n:.1f}%)")
print("=> Cac dong dang co day du o, ty le ngoai lai thap. NHUNG bang van THIEU ngay (khong phai khong khuyet):")
print("   can xu ly khi dung chuoi thoi gian (du bao).")

[2] CHAT LUONG DO LUONG
------------------------------------------------------------
Do phu TB (% gio/ngay co do)   : 97.9%
So ngay phu >= 75%             : 643/657 (97.9%)
Tong so o bi thieu              : 0  (chi tinh cac dong dang co, KHONG tinh ngay vang mat)
So ngay VANG MAT khoi bang      : 43  (2 khoang tram mat du lieu, xem notebook 00)
So ngay cam bien chi do < 50% gio: 8  (trung binh ngay dua tren rat it gio)
Nguong binh thuong (IQR)        : [3.7, 60.0]
So ngay ngoai lai               : 18/657 (2.7%)
=> Cac dong dang co day du o, ty le ngoai lai thap. NHUNG bang van THIEU ngay (khong phai khong khuyet):
   can xu ly khi dung chuoi thoi gian (du bao).


## Bằng chứng 3: Cảm biến có hoạt động đúng

**Câu hỏi trả lời:** không phải "data có đủ" mà là "data có ĐÚNG QUY LUẬT VẬT LÝ không" — nếu cảm biến hỏng, số liệu ra ngẫu nhiên, các mối quan hệ vật lý quen thuộc sẽ biến mất.

**2 quy luật kiểm tra:**
1. Nhiệt độ càng cao → độ ẩm càng thấp (trời nóng thì khô) → tương quan phải **ÂM**
2. Gió càng mạnh → bụi càng ít (gió thổi bay bụi) → tương quan phải **ÂM**

Nếu 2 tương quan này đúng chiều (âm) → cảm biến phản ánh đúng thực tế, đáng tin.

In [3]:
print("[3] NHAT QUAN NOI TAI (quy luat vat ly)")
print("-" * 60)

r_th = df.temperature.corr(df.humidity)   # corr(): tinh he so tuong quan Pearson giua 2 cot
print(f"Tuong quan nhiet do - do am : r = {r_th:.3f}   "
      f"{'DUNG (am, dung quy luat)' if r_th < 0 else 'NGHI NGO!'}")

r_wind = df.pm25.corr(df.wind)
print(f"Tuong quan PM2.5 - gio      : r = {r_wind:.3f}   "
      f"{'DUNG (am, dung quy luat)' if r_wind < 0 else 'NGHI NGO!'}")

print("=> Ca 2 dung chieu vat ly -> cam bien hoat dong on dinh, khong phai so rac")

[3] NHAT QUAN NOI TAI (quy luat vat ly)
------------------------------------------------------------
Tuong quan nhiet do - do am : r = -0.503   DUNG (am, dung quy luat)
Tuong quan PM2.5 - gio      : r = -0.411   DUNG (am, dung quy luat)
=> Ca 2 dung chieu vat ly -> cam bien hoat dong on dinh, khong phai so rac


## Bằng chứng 4: Đối chiếu nguồn ĐỘC LẬP (mô hình CAMS)

**Câu hỏi trả lời:** có nguồn nào KHÁC (không liên quan gì tới OpenAQ) đo cùng vị trí, cùng thời gian, cho ra con số gần giống không?

Đây là bước mạnh nhất, vì nếu 2 nguồn **hoàn toàn độc lập** (khác tổ chức, khác công nghệ đo) mà vẫn khớp nhau — khả năng cả 2 cùng sai là rất thấp. Nguồn đối chiếu ở đây là **CAMS** (Copernicus Atmosphere Monitoring Service — mô hình chất lượng không khí của châu Âu), gọi qua chính Open-Meteo, cùng toạ độ trạm CMT8. **Nguồn:** [Open-Meteo Air Quality API](https://open-meteo.com/en/docs/air-quality-api) (dữ liệu CAMS, lưới toàn cầu khoảng 45 km); **lấy gì:** PM2.5 theo giờ của mô hình; **dùng làm gì:** so với `pm25` của trạm để biết hai nguồn khớp đến đâu (và từ đó biết mức tin cậy khi dùng các khí `_cams` ở notebook 00).


In [4]:
print("[4] DOI CHIEU NGUON DOC LAP (CAMS)")
print("-" * 60)
print("Dang tai du lieu CAMS de so sanh...")

r = requests.get(
    "https://air-quality-api.open-meteo.com/v1/air-quality",
    params={
        "latitude": LAT, "longitude": LON,
        "start_date": str(df["date"].min().date()),
        "end_date": str(df["date"].max().date()),
        "hourly": "pm2_5",          # CAMS tra theo GIO -> phai gop ve ngay o duoi
        "timezone": "Asia/Bangkok",
    },
    timeout=90,
)
h = r.json()["hourly"]
cams = pd.DataFrame({"dt": pd.to_datetime(h["time"]), "pm25_cams": h["pm2_5"]})
cams["date"] = cams["dt"].dt.normalize()          # bo phan gio, chi giu ngay
camsd = cams.groupby("date", as_index=False)["pm25_cams"].mean()   # gom 24 gio/ngay -> 1 trung binh/ngay

j = df[["date", "pm25"]].merge(camsd, on="date", how="inner").dropna()
r_cams = j["pm25"].corr(j["pm25_cams"])
tb_tram, tb_cams = j["pm25"].mean(), j["pm25_cams"].mean()
lech = 100 * (tb_tram - tb_cams) / tb_cams

print(f"So ngay doi chieu duoc          : {len(j)}")
print(f"TB tram CMT8 (do that)          : {tb_tram:.2f} ug/m3")
print(f"TB mo hinh CAMS                 : {tb_cams:.2f} ug/m3")
print(f"Chenh lech muc TB               : {lech:+.1f}%  "
      f"{'-> RAT KHOP' if abs(lech) < 15 else '-> lech dang ke'}")
print(f"Tuong quan ngay-theo-ngay       : r = {r_cams:.3f}")

# Tach theo mua: lech giua mo hinh va tram khong dong deu quanh nam
j = j.merge(df[["date", "mua"]], on="date", how="left")
print()
print("TACH THEO MUA (mo hinh co on dinh quanh nam khong?):")
for ten_mua, nhom in j.groupby("mua"):
    print(f"  {ten_mua}: r = {nhom['pm25'].corr(nhom['pm25_cams']):.2f} | TB tram {nhom['pm25'].mean():.1f} | TB CAMS {nhom['pm25_cams'].mean():.1f}")
cao = j[j["pm25"] > 50]
print(f"  Vao {len(cao)} ngay tram do > 50 ug/m3: tram TB {cao['pm25'].mean():.1f}, CAMS TB {cao['pm25_cams'].mean():.1f}")
j.drop(columns="mua").to_csv("../01_Data/kiem_chung_CMT8_vs_CAMS.csv", index=False)

[4] DOI CHIEU NGUON DOC LAP (CAMS)
------------------------------------------------------------
Dang tai du lieu CAMS de so sanh...


So ngay doi chieu duoc          : 657
TB tram CMT8 (do that)          : 32.38 ug/m3
TB mo hinh CAMS                 : 29.79 ug/m3
Chenh lech muc TB               : +8.7%  -> RAT KHOP
Tuong quan ngay-theo-ngay       : r = 0.310

TACH THEO MUA (mo hinh co on dinh quanh nam khong?):
  Mua kho: r = 0.69 | TB tram 37.2 | TB CAMS 26.8
  Mua mua: r = 0.29 | TB tram 27.9 | TB CAMS 32.6
  Vao 47 ngay tram do > 50 ug/m3: tram TB 59.5, CAMS TB 40.4


**Diễn giải trung thực:**

- **Mức trung bình khớp:** trạm 32,38 so với CAMS 29,79 µg/m³ (lệch +8,7%) → ủng hộ độ lớn của số liệu.
- **Tương quan từng ngày yếu** (r = 0,31) vì CAMS là mô hình toàn cầu lưới thô (~45 km, 0,4°), không bắt được biến động cục bộ của một điểm đo sát đường phố.
- **Mô hình lệch theo mùa:** mùa khô r = 0,69 (trạm 37,2 so với CAMS 26,8), mùa mưa r = 0,29 (trạm 27,9 so với CAMS 32,6); những ngày trạm đo > 50 µg/m³ (47 ngày) trạm TB 59,5 còn CAMS 40,4. Mức trung bình "khớp" là do hai độ lệch ngược chiều bù nhau → các khí `_cams` chỉ dùng để xem xu hướng, không coi là số đo thật.

## Bằng chứng 5: Khớp với tài liệu khoa học đã công bố

**Câu hỏi trả lời:** kết quả có đi ngược lại những gì nghiên cứu khác đã chứng minh không?

**Tài liệu đối chiếu cụ thể (có DOI, tra được thật):**

1. To Thi Hien và cộng sự (2019), *"Current Status of Fine Particulate Matter (PM2.5) in Vietnam's Most Populous City, Ho Chi Minh City"*, **Aerosol and Air Quality Research**, tập 19, tr. 2239–2251.
   DOI: [10.4209/aaqr.2018.12.0471](https://doi.org/10.4209/aaqr.2018.12.0471)
   — Đo 5 điểm tại TP.HCM (03/2017–03/2018), PM2.5 trung bình năm **36.3 ± 13.7 µg/m³**, vượt quy chuẩn VN lúc đó (25) lẫn khuyến nghị WHO lúc đó (10, bản 2005; bản 2021 là 5). Bài nêu (tr.1 và tr.6 của tệp PDF): PM2.5 mùa khô trung bình 39.2 và mùa mưa 33.6 µg/m³ (mùa khô tháng 12–4, mùa mưa tháng 5–11), mức cao nhất vào tháng 11–2 và thấp nhất vào tháng 7–8, PM2.5 tương quan nghịch có ý nghĩa với tốc độ gió. Bài cũng ghi nhận PM2.5 vẫn cao vào các tháng mưa nhiều do khả năng đốt rơm rạ ở Đồng bằng sông Cửu Long. [Link](https://aaqr.org/articles/aaqr-18-12-oa-0471)

2. *"Annual Variation of PM2.5 Chemical Composition in Ho Chi Minh City, Vietnam Including the COVID-19 Outbreak Period"*, **Aerosol and Air Quality Research**.
   DOI: [10.4209/aaqr.220312](https://doi.org/10.4209/aaqr.220312)
   — Đo PM2.5 từ 09/2019 đến 08/2020 (gồm giai đoạn giãn cách COVID-19): trung bình **28.44 ± 11.55 µg/m³**; carbon hữu cơ, carbon nguyên tố và ion tan trong nước chiếm lần lượt 30.7%, 9.7%, 24.9% khối lượng PM2.5 (tr.1). Bài này cũng ghi PM2.5 và tổng ion tan mùa mưa thấp hơn mùa khô, do mưa rửa trôi (tr.1, tr.12), nên dùng được cho cả **độ lớn** lẫn **chiều mùa** (tr.4: trước giãn cách mùa mưa 35.0 và mùa khô 41.1 µg/m³; trong và sau giãn cách COVID-19 mùa khô 24.0 và mùa mưa 22.0 µg/m³). [Link](https://link.springer.com/10.4209/aaqr.220312)

Nếu data của mình cũng cho ra đúng chiều "mùa khô > mùa mưa" (nghiên cứu [1]) và mức PM2.5 cùng cỡ độ lớn (chục µg/m³, vượt chuẩn VN/WHO) (cả [1] và [2]) → khớp với tài liệu, thêm 1 bằng chứng ủng hộ độ tin cậy. Cơ chế rửa trôi bụi bởi mưa được nêu ở T5 (Fujino & Miyamoto 2022), xem bảng tài liệu ở notebook 00.

In [5]:
kho = df[df.mua == "Mua kho"].pm25.mean()
mua = df[df.mua == "Mua mua"].pm25.mean()
tb = df.pm25.mean()
print(f"PM2.5 TB mua kho {kho:.1f} | mua mua {mua:.1f} -> mua kho {'CAO HON' if kho > mua else 'THAP HON'} {100*(kho-mua)/mua:+.0f}%  (To Thi Hien 2019: 39.2 vs 33.6)")
print(f"PM2.5 TB ca ky {tb:.1f} (nghien cuu [1]: 36.3; [2]: 28.44) | gap {tb/5:.1f} lan WHO 2021 (5), {tb/25:.1f} lan QCVN 05:2023 (25)")


PM2.5 TB mua kho 37.2 | mua mua 27.9 -> mua kho CAO HON +33%  (To Thi Hien 2019: 39.2 vs 33.6)
PM2.5 TB ca ky 32.4 (nghien cuu [1]: 36.3; [2]: 28.44) | gap 6.5 lan WHO 2021 (5), 1.3 lan QCVN 05:2023 (25)


## Bằng chứng 6: Nêu rõ hạn chế

Chủ động nêu điểm yếu của dữ liệu mình dùng là tư duy khoa học nghiêm túc, không phải điểm yếu của đồ án.

- **a)** Trạm CMT8 dùng cảm biến **chi phí thấp** (AirGradient), không phải trạm tham chiếu quốc gia → giá trị tuyệt đối có thể lệch.
- **b)** Chưa đối chiếu với trạm chuẩn: trạm Lãnh sự quán Mỹ (ID 7440, cách ~3 km) có ngày cuối 04/03/2025 trên OpenAQ (kết quả Bước A2, notebook 00) → chồng ~3,5 tháng với CMT8; nhóm chưa dùng để đối chiếu.
- **c)** Sensor PM10 của trạm đã ngừng từ 14/01/2025 (notebook 00) → đề tài chỉ dùng được PM2.5 của trạm.
- **d)** Độ ẩm đo được (TB 55,1%) **thấp hơn** độ ẩm ngoài trời thường gặp ở TP.HCM (~70–80% theo số liệu khí tượng) → có thể cảm biến đặt nơi có che chắn. Dùng độ ẩm như biến *tham khảo*.
- **e)** Dữ liệu **một điểm đo**, không đại diện toàn TP.HCM.

## Bảng tổng kết 

In [6]:
# Xay 1 bang tom tat: moi dong la 1 tieu chi, gom ten tieu chi - bang chung - ket qua
# Day chinh la bang de chup man hinh dua vao bao cao
bang = pd.DataFrame([
    ["1. Nguon goc minh bach", "OpenAQ + Open-Meteo, ID tram cong khai", "DAT"],
    ["2. Tai lap duoc", "Notebook day du, chay lai ra cung ket qua", "DAT"],
    ["3. Do phu du lieu", f"{df['pm25_cov'].mean():.1f}% trung binh", "DAT"],
    ["4. Gia tri thieu", f"{thieu} o", "DAT"],
    ["5. Ngoai lai", f"{100*n_out/n:.1f}%", "DAT"],
    ["6. Nhat quan vat ly", f"r(nhiet,am)={r_th:.2f}; r(pm25,gio)={r_wind:.2f}", "DAT"],
    ["7. Doi chieu doc lap", f"Muc TB khop CAMS, lech {lech:+.1f}%", "DAT (1 phan)"],
    ["8. Khop tai lieu KH", "Mua kho > mua mua", "DAT"],
    ["9. Neu ro han che", "5 han che da liet ke", "DAT"],
], columns=["Tieu chi", "Bang chung", "Ket qua"])

bang.to_csv("../01_Data/bang_kiem_chung.csv", index=False)
print("Da luu: ../01_Data/bang_kiem_chung.csv")
bang

Da luu: ../01_Data/bang_kiem_chung.csv


,Tieu chi,Bang chung,Ket qua
0,1. Nguon goc minh bach,"OpenAQ + Open-Meteo, ID tram cong khai",DAT
1,2. Tai lap duoc,"Notebook day du, chay lai ra cung ket qua",DAT
2,3. Do phu du lieu,97.9% trung binh,DAT
3,4. Gia tri thieu,0 o,DAT
4,5. Ngoai lai,2.7%,DAT
5,6. Nhat quan vat ly,"r(nhiet,am)=-0.50; r(pm25,gio)=-0.41",DAT
6,7. Doi chieu doc lap,"Muc TB khop CAMS, lech +8.7%",DAT (1 phan)
7,8. Khop tai lieu KH,Mua kho > mua mua,DAT
8,9. Neu ro han che,5 han che da liet ke,DAT
